In [ ]:
# =============================================================================
# Cell 1 — Imports and Strategy
# =============================================================================
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_ENABLE_AUTO_MIXED_PRECISION'] = '1'

import random, json, math, sys
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Reproducibility
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

# Print versions
print(f"Python  : {sys.version}")
print(f"TF      : {tf.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")

# GPU detection
gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs    : {gpus}")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

# Strategy
if len(gpus) > 1:
    strategy = tf.distribute.MirroredStrategy()
    print("Using MirroredStrategy (multi-GPU)")
elif len(gpus) == 1:
    strategy = tf.distribute.get_strategy()  # default single-GPU
    print("Using default strategy (single GPU)")
else:
    strategy = tf.distribute.get_strategy()
    print("Using default strategy (CPU only)")

REPLICAS = getattr(strategy, 'num_replicas_in_sync', 1)
print(f"Strategy: {strategy.__class__.__name__}, REPLICAS: {REPLICAS}")
print("✅ Cell 1 complete.")

In [ ]:
# =============================================================================
# Cell 2 — Hyperparameters
# =============================================================================
WINDOW             = 599
PER_REPLICA_BATCH  = 128
EPOCHS             = 50
PATIENCE           = 10
SAMPLE_PERIOD      = "5s"          # lowercase 's' — pandas >= 2.0 requirement
MODEL_SAVE_DIR     = "/kaggle/working/seq2point_models"
APPLIANCE          = "fridge"      # ← change this to train other appliances
GLOBAL_BATCH_SIZE  = int(PER_REPLICA_BATCH * REPLICAS)

os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

print(f"WINDOW            = {WINDOW}")
print(f"PER_REPLICA_BATCH = {PER_REPLICA_BATCH}")
print(f"GLOBAL_BATCH_SIZE = {GLOBAL_BATCH_SIZE}")
print(f"EPOCHS            = {EPOCHS}")
print(f"PATIENCE          = {PATIENCE}")
print(f"SAMPLE_PERIOD     = {SAMPLE_PERIOD}")
print(f"MODEL_SAVE_DIR    = {MODEL_SAVE_DIR}")
print(f"APPLIANCE         = {APPLIANCE}")
print(f"REPLICAS          = {REPLICAS}")
print("✅ Cell 2 complete.")

In [ ]:
# =============================================================================
# Cell 3 — Locate Dataset
# =============================================================================
DATA_DIR = None
INPUT_ROOT = "/kaggle/input"

# Strategy 1: look for folder whose path contains 'iawe' or 'i-awe'
for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
    low = dirpath.lower()
    if 'iawe' in low or 'i-awe' in low:
        csv_count = sum(1 for f in filenames if f.endswith('.csv'))
        if csv_count >= 3:
            DATA_DIR = dirpath
            break

# Strategy 2: fallback — folder with most CSV files
if DATA_DIR is None:
    best_dir, best_count = None, 0
    for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
        csv_count = sum(1 for f in filenames if f.endswith('.csv'))
        if csv_count > best_count:
            best_count = csv_count
            best_dir = dirpath
    if best_count >= 3:
        DATA_DIR = best_dir

if DATA_DIR is None:
    print("ERROR: Could not locate dataset.")
    print("Full /kaggle/input tree:")
    for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
        for f in filenames:
            print(f"  {os.path.join(dirpath, f)}")
    raise SystemExit("Dataset not found. Check Kaggle input path.")

print(f"DATA_DIR = {DATA_DIR}")
print()
print("Full recursive file tree:")
for dirpath, dirnames, filenames in os.walk(DATA_DIR):
    level = dirpath.replace(DATA_DIR, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(dirpath)}/")
    sub_indent = '  ' * (level + 1)
    for f in sorted(filenames):
        fsize = os.path.getsize(os.path.join(dirpath, f))
        print(f"{sub_indent}{f}  ({fsize / 1024:.1f} KB)")
print()
print("✅ Cell 3 complete.")

In [ ]:
# =============================================================================
# Cell 4 — Helper Functions
# =============================================================================

# iAWE channel-to-appliance mapping (file stem → appliance)
IAWE_APPLIANCE_MAP = {
    "aggregate": ["1"], "mains": ["1"],
    "ac": ["2"], "air conditioner": ["2"],
    "fridge": ["5"], "refrigerator": ["5"],
    "washing machine": ["6"], "washer": ["6"],
    "laptop": ["7"],
    "television": ["11"], "tv": ["11"],
    "geyser": ["13"],
}

# Null markers that appear in iAWE CSVs
_NULL_VALUES = ["\\N", "\\\\N", "NULL", "null", "NA", "N/A",
                "n/a", "", "nan", "NaN"]


def get_iawe_ids(name):
    """Return list of iAWE file-stem IDs for a given appliance name."""
    key = name.lower().strip()
    if key in IAWE_APPLIANCE_MAP:
        return IAWE_APPLIANCE_MAP[key]
    return []


def find_file_recursive(folder, keywords, iawe_ids=None):
    """
    Walk *folder* recursively. Return first .csv whose:
      - filename contains any keyword, OR
      - file stem (without .csv) is in iawe_ids set, OR
      - parent folder name is in iawe_ids set.
    """
    iawe_set = set(iawe_ids) if iawe_ids else set()
    for dirpath, _, filenames in os.walk(folder):
        for fname in sorted(filenames):
            if not fname.lower().endswith('.csv'):
                continue
            low = fname.lower()
            stem = os.path.splitext(fname)[0]  # e.g. '5' from '5.csv'
            parent = os.path.basename(dirpath)
            # Check keywords
            for kw in keywords:
                if kw in low:
                    return os.path.join(dirpath, fname)
            # Check iAWE IDs
            if stem in iawe_set or parent in iawe_set:
                return os.path.join(dirpath, fname)
    return None


def load_csv_timeseries(path):
    """
    Load iAWE-style CSV into a pandas Series with DatetimeIndex.
    Attempt 1: headerless (col0=unix_ts, col1=watts).
    Attempt 2: headered CSV, first column as index.
    """
    # ---- Attempt 1: headerless CSV ----
    try:
        df = pd.read_csv(path, header=None, dtype=str,
                         na_values=_NULL_VALUES, low_memory=False)
        ts_raw = pd.to_numeric(df.iloc[:, 0], errors='coerce')
        pw_raw = pd.to_numeric(df.iloc[:, 1], errors='coerce')
        valid = ts_raw.notna() & pw_raw.notna()
        if valid.sum() > 10:
            ts = pd.to_datetime(ts_raw[valid], unit='s')
            s = pd.Series(pw_raw[valid].values, index=ts,
                          dtype=np.float64)
            s = s.sort_index()
            s = s[~s.index.duplicated(keep='first')]
            print(f"  Loaded (headerless): {os.path.basename(path)} "
                  f"— {len(s)} rows, "
                  f"{s.index.min()} → {s.index.max()}")
            return s
    except Exception:
        pass

    # ---- Attempt 2: headered CSV ----
    try:
        df2 = pd.read_csv(path, header=0, dtype=str,
                          na_values=_NULL_VALUES, low_memory=False,
                          index_col=0)
        df2.index = pd.to_datetime(df2.index, errors='coerce')
        df2 = df2[df2.index.notna()]
        for col in df2.columns:
            vals = pd.to_numeric(df2[col], errors='coerce')
            if vals.notna().sum() > 10:
                s = vals.dropna().sort_index()
                s = s[~s.index.duplicated(keep='first')]
                print(f"  Loaded (headered): {os.path.basename(path)} "
                      f"col='{col}' — {len(s)} rows")
                return s
    except Exception:
        pass

    raise ValueError(
        f"Cannot parse CSV timeseries: {path}\n"
        f"Tried headerless (unix_ts, watts) and headered formats.")


print("Helper functions defined:")
print("  IAWE_APPLIANCE_MAP :", list(IAWE_APPLIANCE_MAP.keys()))
print("  get_iawe_ids()")
print("  find_file_recursive()")
print("  load_csv_timeseries()")
print("✅ Cell 4 ready.")

In [ ]:
# =============================================================================
# Cell 5 — Load, Resample, Align, Normalize
# =============================================================================
print(f"Training Seq2Point for: {APPLIANCE}")
print("=" * 60)

# --- Find files ---
agg_path = find_file_recursive(
    DATA_DIR,
    keywords=["aggregate", "mains", "main", "agg"],
    iawe_ids=["1"]
)
app_ids = get_iawe_ids(APPLIANCE)
app_path = find_file_recursive(
    DATA_DIR,
    keywords=[APPLIANCE, APPLIANCE[:3]],
    iawe_ids=app_ids
)

print(f"Aggregate file : {agg_path}")
print(f"Appliance file : {app_path}")

if agg_path is None or app_path is None:
    print("\nERROR: file not found. Available CSVs:")
    for dp, _, fns in os.walk(DATA_DIR):
        for fn in fns:
            if fn.endswith('.csv'):
                print(f"  {os.path.join(dp, fn)}")
    raise SystemExit("Fix file paths and re-run.")

# --- Load ---
print("\nLoading aggregate...")
agg_raw = load_csv_timeseries(agg_path)
print("Loading appliance...")
app_raw = load_csv_timeseries(app_path)

# --- Resample to 5s ---
print(f"\nResampling to {SAMPLE_PERIOD}...")
agg = agg_raw.resample(SAMPLE_PERIOD).mean().interpolate(
    method='linear', limit=5)
app = app_raw.resample(SAMPLE_PERIOD).mean().interpolate(
    method='linear', limit=5)
print(f"  Aggregate after resample: {len(agg)}")
print(f"  Appliance after resample: {len(app)}")

# --- Align ---
common_idx = agg.index.intersection(app.index)
agg = agg.loc[common_idx]
app = app.loc[common_idx]
N = len(agg)
days = N * 5 / 86400  # 5s per sample
print(f"\nAligned samples : {N:,}")
print(f"Approx duration : {days:.1f} days")

if N <= WINDOW:
    raise SystemExit(
        f"Not enough samples ({N}) for WINDOW={WINDOW}. "
        f"Check data or reduce WINDOW.")

# --- Normalize ---
agg_mean = float(agg.mean())
agg_std  = float(agg.std()) if float(agg.std()) > 0 else 1.0
app_mean = float(app.mean())
app_std  = float(app.std()) if float(app.std()) > 0 else 1.0

agg_norm = ((agg.values - agg_mean) / agg_std).astype(np.float32)
app_norm = ((app.values - app_mean) / app_std).astype(np.float32)

# Save normalization params
params = {
    "agg_mean": agg_mean, "agg_std": agg_std,
    "app_mean": app_mean, "app_std": app_std,
}
params_path = os.path.join(MODEL_SAVE_DIR,
                           f"seq2point_{APPLIANCE}_params.json")
with open(params_path, 'w') as f:
    json.dump(params, f, indent=2)

print(f"\nagg_mean={agg_mean:.2f}, agg_std={agg_std:.2f}")
print(f"app_mean={app_mean:.2f}, app_std={app_std:.2f}")
print(f"Params saved → {params_path}")

# --- Window splits ---
N_windows = N - WINDOW
train_end = int(N_windows * 0.8)
val_end   = int(N_windows * 0.9)

print(f"\nN_windows = {N_windows:,}")
print(f"train_end = {train_end:,}  (80%)")
print(f"val_end   = {val_end:,}  (90%)")
print(f"test      = {N_windows - val_end:,}  (10%)")
print("✅ Cell 5 complete.")

In [ ]:
# =============================================================================
# Cell 6 — NaN / Inf Audit and Cleanup
# =============================================================================
print("BEFORE cleanup:")
print(f"  agg_norm — NaN: {np.isnan(agg_norm).sum()}, "
      f"Inf: {np.isinf(agg_norm).sum()}, "
      f"range: [{np.nanmin(agg_norm):.3f}, {np.nanmax(agg_norm):.3f}]")
print(f"  app_norm — NaN: {np.isnan(app_norm).sum()}, "
      f"Inf: {np.isinf(app_norm).sum()}, "
      f"range: [{np.nanmin(app_norm):.3f}, {np.nanmax(app_norm):.3f}]")


def clean_norm(arr):
    """Forward-fill, back-fill, zero-fill any remaining NaN, then clip."""
    s = pd.Series(arr)
    s = s.ffill().bfill().fillna(0.0)
    out = s.values.astype(np.float32)
    out = np.clip(out, -3.0, 3.0)
    return out


agg_norm = clean_norm(agg_norm)
app_norm = clean_norm(app_norm)

print("\nAFTER cleanup:")
print(f"  agg_norm — NaN: {np.isnan(agg_norm).sum()}, "
      f"range: [{agg_norm.min():.3f}, {agg_norm.max():.3f}]")
print(f"  app_norm — NaN: {np.isnan(app_norm).sum()}, "
      f"range: [{app_norm.min():.3f}, {app_norm.max():.3f}]")

# Hard assertions — will stop the notebook if any fail
assert np.isnan(agg_norm).sum() == 0, "agg_norm still has NaN!"
assert np.isnan(app_norm).sum() == 0, "app_norm still has NaN!"
assert np.isinf(agg_norm).sum() == 0, "agg_norm has Inf!"
assert np.isinf(app_norm).sum() == 0, "app_norm has Inf!"

print("\n✅ Data clean. Proceed to Cell 7.")

In [ ]:
# =============================================================================
# Cell 7 — Build tf.data Datasets
# =============================================================================

def gen_windows(start, end):
    """Yield (X, y) pairs for window indices [start, end)."""
    for i in range(start, end):
        X = agg_norm[i : i + WINDOW].reshape(WINDOW, 1)             # (599,1)
        y = np.array([app_norm[i + WINDOW // 2]], dtype=np.float32)  # (1,)
        yield X, y


output_signature = (
    tf.TensorSpec(shape=(WINDOW, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(1,),        dtype=tf.float32),
)


def make_ds(start, end, batch_size, shuffle=False):
    """Create a repeating, batched tf.data.Dataset from the generator."""
    ds = tf.data.Dataset.from_generator(
        lambda s=start, e=end: gen_windows(s, e),
        output_signature=output_signature,
    )
    if shuffle:
        ds = ds.shuffle(10000, seed=42, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.repeat()           # ← CRITICAL for MirroredStrategy generators
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


# Sizes
train_size = train_end
val_size   = val_end - train_end
test_size  = N_windows - val_end

n_train_steps = train_size // GLOBAL_BATCH_SIZE
n_val_steps   = val_size   // GLOBAL_BATCH_SIZE
n_test_steps  = test_size  // GLOBAL_BATCH_SIZE

# Build datasets
train_ds = make_ds(0,         train_end, GLOBAL_BATCH_SIZE, shuffle=True)
val_ds   = make_ds(train_end, val_end,   GLOBAL_BATCH_SIZE, shuffle=False)
test_ds  = make_ds(val_end,   N_windows, GLOBAL_BATCH_SIZE, shuffle=False)

print(f"train_size  = {train_size:,}  →  steps/epoch = {n_train_steps}")
print(f"val_size    = {val_size:,}  →  val_steps   = {n_val_steps}")
print(f"test_size   = {test_size:,}  →  test_steps  = {n_test_steps}")
print(f"GLOBAL_BATCH_SIZE = {GLOBAL_BATCH_SIZE}")
print("✅ Cell 7 complete.")

In [ ]:
# =============================================================================
# Cell 8 — Build Model
# =============================================================================
from tensorflow.keras import mixed_precision, regularizers

# Force float32 — mixed_float16 causes NaN with this data distribution
mixed_precision.set_global_policy('float32')
print(f"Precision policy: {mixed_precision.global_policy().name}")

L2 = 1e-4


def build_seq2point(input_length):
    """Seq2Point Conv1D model with BatchNorm and regularization."""
    inp = layers.Input(shape=(input_length, 1), dtype='float32')

    # Block 1
    x = layers.Conv1D(16, 10, padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(L2))(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Block 2
    x = layers.Conv1D(16, 8, padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(L2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Block 3
    x = layers.Conv1D(32, 6, padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(L2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Block 4
    x = layers.Conv1D(32, 5, padding='same', use_bias=False,
                      kernel_regularizer=regularizers.l2(L2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    # Head
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, use_bias=False,
                     kernel_regularizer=regularizers.l2(L2))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.5)(x)

    out = layers.Dense(1, dtype='float32')(x)
    return keras.Model(inp, out, name='seq2point')


# Build and compile inside strategy scope
with strategy.scope():
    model = build_seq2point(WINDOW)
    optimizer = keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0)
    model.compile(
        optimizer=optimizer,
        loss=keras.losses.Huber(delta=1.0),
        metrics=['mae'],
    )

model.summary()
print("\n✅ Cell 8 complete.")

In [ ]:
# =============================================================================
# Cell 9 — Train Model
# =============================================================================

class StopOnNaN(callbacks.Callback):
    """Stop training immediately if loss becomes NaN or Inf."""
    def on_batch_end(self, batch, logs=None):
        logs = logs or {}
        loss = logs.get('loss')
        if loss is not None and not np.isfinite(loss):
            print(f"\n⚠️ NaN/Inf loss at batch {batch}. Stopping.")
            self.model.stop_training = True


ckpt_path = os.path.join(MODEL_SAVE_DIR,
                         f"seq2point_{APPLIANCE}_best.keras")
csv_log   = os.path.join(MODEL_SAVE_DIR,
                         f"training_history_{APPLIANCE}.csv")

cb = [
    callbacks.ModelCheckpoint(
        ckpt_path, monitor='val_loss',
        save_best_only=True, verbose=1),
    callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=4, min_lr=1e-7, verbose=1),
    callbacks.CSVLogger(csv_log, append=False),
    StopOnNaN(),
]

print(f"Training for up to {EPOCHS} epochs...")
print(f"  steps_per_epoch  = {n_train_steps}")
print(f"  validation_steps = {n_val_steps}")
print(f"  checkpoint       = {ckpt_path}")
print()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=n_train_steps,
    validation_steps=n_val_steps,
    callbacks=cb,
)

print("\n✅ Cell 9 complete — training finished.")

In [ ]:
# =============================================================================
# Cell 10 — Evaluate, Save, Inference Check
# =============================================================================

# --- Evaluate on test set ---
print("Evaluating on test set...")
test_loss, test_mae = model.evaluate(test_ds, steps=n_test_steps, verbose=1)
print(f"\nTest Huber loss (norm) : {test_loss:.6f}")
print(f"Test MAE (norm)        : {test_mae:.6f}")
print(f"Test MAE (watts)       : {test_mae * app_std:.2f} W")

# --- Save final model ---
final_path = os.path.join(MODEL_SAVE_DIR,
                          f"seq2point_{APPLIANCE}_final.keras")
model.save(final_path)
print(f"\nFinal model saved → {final_path}")

# --- Save training history ---
hist_path = os.path.join(MODEL_SAVE_DIR,
                         f"training_history_{APPLIANCE}_full.csv")
hist_df = pd.DataFrame(history.history)
hist_df.to_csv(hist_path, index=False)
print(f"History saved    → {hist_path}")
print("\nLast 5 epochs:")
print(hist_df.tail(5).to_string(index=True))

# --- Quick inference sanity check ---
print("\n" + "=" * 60)
print("Inference sanity check")
print("=" * 60)
idx = random.randint(val_end, N_windows - 1)
window = agg_norm[idx : idx + WINDOW].reshape(1, WINDOW, 1)
pred_norm   = float(model.predict(window, verbose=0)[0, 0])
pred_watts  = pred_norm * app_std + app_mean
actual_norm = float(app_norm[idx + WINDOW // 2])
actual_watts = actual_norm * app_std + app_mean
error = abs(pred_watts - actual_watts)
print(f"Window index    : {idx}")
print(f"Predicted watts : {pred_watts:.2f} W")
print(f"Actual watts    : {actual_watts:.2f} W")
print(f"Absolute error  : {error:.2f} W")

# --- List all saved files ---
print(f"\nFiles in {MODEL_SAVE_DIR}:")
for f in sorted(os.listdir(MODEL_SAVE_DIR)):
    fp = os.path.join(MODEL_SAVE_DIR, f)
    if os.path.isfile(fp):
        size_kb = os.path.getsize(fp) / 1024
        print(f"  {f:<50s}  {size_kb:>8.1f} KB")

print("\n✅ Cell 10 complete — all done!")